In [ ]:
import json
import gzip
from datetime import datetime, date, timedelta
import pandas as pd
import boto3
import pytz
import time


def convert_to_ist(timestamp):
    # Ensure it's a pandas Timestamp and convert to UTC timezone
    if isinstance(timestamp, pd.Timestamp):
        # Convert to UTC if necessary
        timestamp = timestamp.tz_localize('UTC') if timestamp.tzinfo is None else timestamp
        # Convert to IST
        dt_ist = timestamp.astimezone(pytz.timezone('Asia/Kolkata'))
        return dt_ist
    else:
        raise TypeError("Expected a pandas Timestamp")



# Function to convert milliseconds to timestamp
def convert_to_timestamp(milliseconds):
    timestamp_s = int(milliseconds) / 1000
    human_readable_timestamp = datetime.utcfromtimestamp(timestamp_s).strftime('%Y-%m-%d %H:%M:%S')
    return human_readable_timestamp


def get_data(platform, prev_date):
    
    d = datetime.strptime(prev_date, '%m-%d-%Y')
    d = date.strftime(d, "%m-%d-%Y")
    ##Got the data from another region
    bucket_name = ""
    aws_access_key_id = ""
    aws_secret_access_key = ""
    region_name = ""
    
    session = boto3.Session(aws_access_key_id=aws_access_key_id, aws_secret_access_key=aws_secret_access_key, region_name=region_name)
    s3 = session.resource('s3')
    
    path = 'clickstream/rudder-logs/2RYrpPd26coeWB7cRSZI2ihdNAX/' + prev_date + '/'
    my_bucket = s3.Bucket(bucket_name)
    
    final_data = []
    
    
    for my_bucket_object in my_bucket.objects.filter(Prefix=path):
        key_name = my_bucket_object.key
        obj = s3.Object(bucket_name, key_name)
        data = gzip.decompress(obj.get()['Body'].read())
        data = data.decode('utf-8')
        partitioned_string = data.split('\n')
        
        records = []
        c = 0
        
        for string in partitioned_string:
            if len(string) == 0:
                continue
            else:
                try:
                    records.append(json.loads(string))
                except:
                    print("error")
                    print(c)
                    print(string[-2:])
                c += 1

        for rec in records:
            try:
                if rec['type'] == 'track':
                    event_dict = {}
                    if 'event' in rec:
                        event_dict['event'] = rec['event']
                        if 'context' in rec:
                            if 'traits' in rec['context']:
                                event_dict['email'] = rec['context']['traits'].get('email', '')
                            else:
                                event_dict['email'] = ''
                        else:
                            event_dict['email'] = ''

                        if 'anonymousId' in rec:
                            event_dict['device'] = rec['anonymousId']
                        if 'originalTimestamp' in rec:
                            p = '%Y-%m-%dT%H:%M:%S.%fZ'
                            mytime = rec['originalTimestamp']
                            epoch = datetime(1970, 1, 1)
                            t = (datetime.strptime(mytime, p) - epoch).total_seconds()
                            t = str(int(t*1000))
                            event_dict['time'] = t
                        final_data.append(event_dict)
            except:
                print(rec)

    df = pd.DataFrame(final_data)
    print("data reading done")
    print(df['time'][0])
    df['timestamp'] = df['time'].apply(convert_to_timestamp)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    date_filter = datetime.strptime(prev_date, '%m-%d-%Y')

    print(df)
    filtered_df = df[df['timestamp'].dt.date == pd.to_datetime(date_filter).date()]
    df = filtered_df.copy()
#     print(df['time'][0])
    df['event'] = df['event'].apply(lambda x: [x])
    
    grouped_df = df.groupby(['email', 'device'], as_index=False).agg({'time': ['min', 'max'], 'event': 'sum'})
    grouped_df.columns = ['email', 'device', 'first_time', 'last_time', 'sequence']
    grouped_df = grouped_df[['device', 'email', 'sequence', 'first_time', 'last_time']]

    # Define a function to convert timestamp in milliseconds to datetime
    grouped_df['email'] = grouped_df.groupby('device')['email'].ffill().bfill()
    #grouped_df.to_csv("data_grouped.csv")
    

    print("starting to write")
    # writing in s3
    bucket_name = ""
    aws_access_key_id = ""
    aws_secret_access_key = ""
    region_name = ""
    
    s3_client = boto3.client('s3', aws_access_key_id= aws_access_key_id, aws_secret_access_key = aws_secret_access_key, region_name = region_name)
    result = grouped_df.to_json(orient="records")
    bytes_to_write = result.encode()
    filename = ""
    s3_client.put_object(Body=bytes_to_write, Bucket=bucket_name, Key=filename)
    print("written")

if __name__ == "__main__":
    start_time = time.time()
    curr_date = datetime.today().strftime('%m-%d-%Y')
    prev_date = (datetime.today() - timedelta(days=1)).strftime('%m-%d-%Y')
    get_data('web', curr_date)
    end_time = time.time()
    print("Time Taken: " )
    print((end_time - start_time))



In [ ]:
[
  {
    "device": "abc123",
    "email": "user@gmail.com",
    "sequence": ["Login","Buy"],
    "first_time": "...",
    "last_time": "..."
  }
]